In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
df = pd.read_csv("../datasets/processed/merged_data.csv")
print(df.head())

df = df.dropna(subset=["asylum_applications"]).copy()
 
numeric_features = [
    "year", "gdp_per_capita", "unemployment_rate", "population",
    "urban_pct", "temp_mean", "heatwave_days", "precip_total",
    "precip_days_heavy", "dry_days", "evapotrans_total",
]
for col in numeric_features:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
 
df = pd.get_dummies(df, columns=["country_code"], drop_first=True)
country_cols = [c for c in df.columns if c.startswith("country_code_")]
features = numeric_features + country_cols
 
df["log_asylum"] = np.log1p(df["asylum_applications"])
 
train = df[df["year"] <= 2018]
test  = df[df["year"] >= 2019]
print(f"Train rows: {len(train)} | Test rows: {len(test)}")
 
X_train, y_train = train[features], train["log_asylum"]
X_test,  y_test  = test[features],  test["log_asylum"]
 
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
 
mse_log = mean_squared_error(y_test, y_pred)
r2_log  = r2_score(y_test, y_pred)
print("MSE (log scale):", mse_log)
print("R² (log scale):", r2_log)
 
y_pred_orig = np.expm1(y_pred)
y_test_orig = np.expm1(y_test)
mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)
print(f"MAE (original count scale): {mae_orig:,.0f}")
 
new_row = pd.DataFrame([{
    "year": 2015,
    "gdp_per_capita": 52000,
    "unemployment_rate": 5.1,
    "population": 8600000,
    "urban_pct": 67.5,
    "temp_mean": 11.2,
    "heatwave_days": 1,
    "precip_total": 700,
    "precip_days_heavy": 3,
    "dry_days": 240,
    "evapotrans_total": 820,
    "country_code": "AT",
}])
new_row = pd.get_dummies(new_row, columns=["country_code"])
new_row = new_row.reindex(columns=features, fill_value=0)
 
log_prediction = model.predict(new_row)
prediction = np.expm1(log_prediction)
print("Predicted asylum applications:", round(prediction[0]))

results = test[["year"]].copy()
results["country"] = test[country_cols].idxmax(axis=1).str.replace("country_code_", "")
results["actual"] = y_test_orig.values
results["predicted"] = y_pred_orig
results["error"] = results["actual"] - results["predicted"]
print(results.sort_values("error", key=abs, ascending=False).head(15))


   year  gdp_per_capita  unemployment_rate  population  urban_pct  \
0  2010    46611.139342              4.883   8363404.0  67.104743   
1  2011    51116.895352              4.637   8391643.0  67.148426   
2  2012    48250.405914              4.909   8429991.0  67.208591   
3  2013    50305.354577              5.367   8479823.0  67.298131   
4  2014    51314.972262              5.674   8546356.0  67.415433   

  country_code  asylum_applications  temp_mean  heatwave_days  precip_total  \
0           AT              11060.0   9.422351              0         783.7   
1           AT              14455.0  10.900239              0         498.1   
2           AT              17450.0  11.087264              0         552.6   
3           AT              17520.0  10.479536              2         709.0   
4           AT              28065.0  11.703919              0         796.7   

   precip_days_heavy  dry_days  evapotrans_total  
0                  7       229         759.99400  
1       